# P1: anomaly detection benchmark on MVTec AD

PatchCore and PaDiM, each run with two backbones (Wide-ResNet50 and ResNet18), on all 15 MVTec AD categories.
This is a 2x2 design (method x backbone), so the notebook can separate how much of the performance gap comes from the **method** and how much from the **backbone**.

**How to use:** run the cells top to bottom. The notebook only reads `metrics/mvtec_metrics.csv`; it never trains anything. To refresh after new runs, rerun the whole notebook (the prose findings at the end are written for the 17 Oct 2026 run and should be rechecked against the tables).

**Metric notes**
- AUROC needs no threshold, so it is the headline metric. F1 depends on a threshold that anomalib sets automatically, so treat it with caution.
- Image AUROC measures whether a defective image is ranked above a normal one. Pixel AUROC measures how well the anomaly map highlights the defect.
- One seed per run, so differences below about 0.01 are not meaningful.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

CANDIDATES = [
    Path(r"D:\PORTFOLIO_2026\P1_defect_detection\metrics\mvtec_metrics.csv"),
    Path.cwd().parent / "metrics" / "mvtec_metrics.csv",
    Path.cwd() / "mvtec_metrics.csv",
]
METRICS_CSV = next((p for p in CANDIDATES if p.exists()), None)
if METRICS_CSV is None:
    raise FileNotFoundError("mvtec_metrics.csv not found. Tried:\n" + "\n".join(map(str, CANDIDATES)))

FIG_DIR = METRICS_CSV.parent.parent / "figures"
FIG_DIR.mkdir(exist_ok=True)
print("Reading:", METRICS_CSV)
print("Figures are saved to:", FIG_DIR)

SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
METHOD_COLOR = {"PatchCore": "#2a78d6", "PaDiM": "#eb6834"}
EFFECT_COLOR = {"method": "#6250d6", "backbone": "#008300"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": ["Segoe UI", "Arial", "DejaVu Sans"], "font.size": 10.5,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.dpi": 110,
})

## 0. Load and check the data

In [ ]:
VARIANTS = {                       # model name in the CSV -> (method, backbone)
    "patchcore":     ("PatchCore", "WRN50"),
    "patchcore_r18": ("PatchCore", "R18"),
    "padim":         ("PaDiM",     "R18"),
    "padim_wrn50":   ("PaDiM",     "WRN50"),
}
VARIANT_OF = {v: k for k, v in VARIANTS.items()}
TEXTURES = {"carpet", "grid", "leather", "tile", "wood"}

raw = pd.read_csv(METRICS_CSV)
unknown = set(raw["model"]) - set(VARIANTS)
assert not unknown, f"Unexpected model names in the CSV: {unknown}"
assert not raw.duplicated(["model", "category"]).any(), "Duplicate (model, category) rows in the CSV"
counts = raw.groupby("model").size()
assert (counts == 15).all() and len(counts) == 4, f"Expected 4 variants x 15 categories, got:\n{counts}"

df = raw.copy()
df["method"] = df["model"].map(lambda m: VARIANTS[m][0])
df["backbone"] = df["model"].map(lambda m: VARIANTS[m][1])
df["group"] = np.where(df["category"].isin(TEXTURES), "texture", "object")
print(f"{len(df)} rows: 4 variants x 15 categories, no duplicates")

def wide(metric):
    # one row per category, one column per model name
    return df.pivot_table(index="category", columns="model", values=metric)

img, pix = wide("image_AUROC"), wide("pixel_AUROC")

## 1. The 2x2 result

Mean over the 15 categories. Rows are methods, columns are backbones.
PaDiM with Wide-ResNet50 uses 100 features (not the paper's 550) because the full setting does not fit in 8 GB of GPU memory.

In [ ]:
def two_by_two(metric):
    t = df.pivot_table(index="method", columns="backbone", values=metric)
    return t.loc[["PatchCore", "PaDiM"], ["WRN50", "R18"]].round(3)

for metric, label in [("image_AUROC", "Image AUROC"), ("pixel_AUROC", "Pixel AUROC"), ("pixel_F1Score", "Pixel F1")]:
    print(label)
    display(two_by_two(metric))

## 2. Method effect vs backbone effect

All values are differences in image AUROC. A positive method effect means PatchCore beats PaDiM.

In [ ]:
effects = pd.DataFrame({
    "method_gap_WRN50":        img["patchcore"]     - img["padim_wrn50"],
    "method_gap_R18":          img["patchcore_r18"] - img["padim"],
    "backbone_gain_PatchCore": img["patchcore"]     - img["patchcore_r18"],
    "backbone_gain_PaDiM":     img["padim_wrn50"]   - img["padim"],
})
ORDER = effects.sort_values("method_gap_WRN50", ascending=False).index.tolist()

display(effects.loc[ORDER].round(3))

print("Mean over categories:")
display(effects.mean().round(3).to_frame("mean").T)

print(f"PatchCore beats PaDiM on image AUROC: {(effects.method_gap_WRN50 > 0).sum()}/15 at WRN50, "
      f"{(effects.method_gap_R18 > 0).sum()}/15 at R18")
print(f"Pixel AUROC, PatchCore beats PaDiM: {(pix['patchcore'] > pix['padim_wrn50']).sum()}/15 at WRN50, "
      f"{(pix['patchcore_r18'] > pix['padim']).sum()}/15 at R18")
print(f"Bigger backbone helps PaDiM on {(effects.backbone_gain_PaDiM > 0).sum()}/15 categories, "
      f"hurts on {(effects.backbone_gain_PaDiM < 0).sum()}/15")

## 3. Every category, all four variants

Color shows the method, fill shows the backbone (filled = Wide-ResNet50, hollow = ResNet18). The line joins the two backbones of the same method, so its length is the backbone effect. Categories are sorted by the method gap at Wide-ResNet50, largest at the top.

In [ ]:
def legend_handles():
    h = [Line2D([], [], marker="o", ls="", ms=8, mfc=c, mec=c, label=m) for m, c in METHOD_COLOR.items()]
    h += [Line2D([], [], marker="o", ls="", ms=8, mfc=INK2, mec=SURFACE, label="Wide-ResNet50 (filled)"),
          Line2D([], [], marker="o", ls="", ms=8, mfc=SURFACE, mec=INK2, mew=1.8, label="ResNet18 (hollow)")]
    return h

def dot_panel(ax, table, order, xlim, xlabel):
    # table: rows = labels, columns = model names
    ys = np.arange(len(order))[::-1].astype(float)
    off = {"PatchCore": 0.17, "PaDiM": -0.17}
    ax.set_axisbelow(True)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    for method, color in METHOD_COLOR.items():
        r18 = table.loc[order, VARIANT_OF[(method, "R18")]].to_numpy()
        wrn = table.loc[order, VARIANT_OF[(method, "WRN50")]].to_numpy()
        for yy, lo, hi in zip(ys + off[method], r18, wrn):
            ax.plot([lo, hi], [yy, yy], color=color, lw=1.4, alpha=0.4, solid_capstyle="round", zorder=2)
        ax.scatter(r18, ys + off[method], s=64, facecolor=SURFACE, edgecolor=color, linewidth=1.8, zorder=3)
        ax.scatter(wrn, ys + off[method], s=64, facecolor=color, edgecolor=SURFACE, linewidth=1.0, zorder=4)
    ax.set_yticks(ys)
    ax.set_yticklabels(order)
    ax.set_xlim(*xlim)
    ax.set_ylim(-0.55, len(order) - 0.45)
    ax.set_xlabel(xlabel)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", length=3)

def category_figure(metric, xlim, fname, title):
    fig, ax = plt.subplots(figsize=(8.6, 6.2))
    dot_panel(ax, wide(metric), ORDER, xlim, {"image_AUROC": "Image AUROC", "pixel_AUROC": "Pixel AUROC"}[metric])
    ax.legend(handles=legend_handles(), loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4, frameon=False,
              fontsize=9.5, labelcolor=INK2, handletextpad=0.3, columnspacing=1.4)
    ax.set_title(title, loc="left", pad=30, fontsize=12.5, color=INK, weight="medium")
    fig.savefig(FIG_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()

category_figure("image_AUROC", (0.66, 1.01), "image_auroc_by_category.png", "Image AUROC per category")

In [ ]:
category_figure("pixel_AUROC", (0.89, 1.0), "pixel_auroc_by_category.png", "Pixel AUROC per category")

## 4. How much is the method, how much is the backbone?

For each category, the purple bar is the gain from switching PaDiM to PatchCore (both on Wide-ResNet50). The green bar is the gain PaDiM gets from the bigger backbone. Labels are shown for effects of 0.10 or more in size.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 6.4))
ys = np.arange(len(ORDER))[::-1].astype(float)
h, gap = 0.34, 0.02
method = effects.loc[ORDER, "method_gap_WRN50"].to_numpy()
backbone = effects.loc[ORDER, "backbone_gain_PaDiM"].to_numpy()

ax.set_axisbelow(True)
ax.xaxis.grid(True, color=GRID, lw=0.8)
ax.axvline(0, color=AXIS, lw=1.0, zorder=1)
ax.barh(ys + h / 2 + gap, method, height=h, color=EFFECT_COLOR["method"],
        label="Method effect: PatchCore minus PaDiM (both Wide-ResNet50)")
ax.barh(ys - h / 2 - gap, backbone, height=h, color=EFFECT_COLOR["backbone"],
        label="Backbone effect: Wide-ResNet50 minus ResNet18 (PaDiM)")

for yy, v in list(zip(ys + h / 2 + gap, method)) + list(zip(ys - h / 2 - gap, backbone)):
    if abs(v) >= 0.10:
        ax.text(v + (0.006 if v > 0 else -0.006), yy, f"{v:+.2f}", va="center",
                ha="left" if v > 0 else "right", fontsize=8.5, color=INK2)

ax.set_yticks(ys)
ax.set_yticklabels(ORDER)
ax.set_xlim(-0.11, 0.33)
ax.set_xlabel("Change in image AUROC")
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", length=3)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=1, frameon=False, fontsize=9.5, labelcolor=INK2)
ax.set_title("Method effect vs backbone effect, per category", loc="left", pad=48, fontsize=12.5, color=INK, weight="medium")
fig.savefig(FIG_DIR / "method_vs_backbone_effect.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Textures vs objects

Same encoding as section 3, averaged over the 5 texture categories (carpet, grid, leather, tile, wood) and the 10 object categories.

In [ ]:
def grouped(metric):
    t = df.pivot_table(index="group", columns="model", values=metric)
    return t.rename(index={"texture": "texture (5)", "object": "object (10)"})

rows = ["texture (5)", "object (10)"]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
fig.subplots_adjust(left=0.1, right=0.98, top=0.80, bottom=0.2, wspace=0.3)
dot_panel(axes[0], grouped("image_AUROC"), rows, (0.84, 1.0), "Image AUROC (mean over categories)")
dot_panel(axes[1], grouped("pixel_AUROC"), rows, (0.93, 1.0), "Pixel AUROC (mean over categories)")
fig.legend(handles=legend_handles(), loc="lower left", bbox_to_anchor=(0.06, 0.82), ncol=4, frameon=False,
           fontsize=9.5, labelcolor=INK2, handletextpad=0.3, columnspacing=1.4)
fig.text(0.06, 0.995, "Textures vs objects", ha="left", va="top", fontsize=12.5, color=INK, weight="medium")
fig.savefig(FIG_DIR / "texture_vs_object.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Export a README-ready table

In [ ]:
def md_table(frame, title):
    cols = [title] + list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in frame.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [f"{v:.3f}" for v in row]) + " |")
    return "\n".join(lines)

readme_tables = "\n\n".join([
    "Mean image AUROC (15 MVTec AD categories)\n\n" + md_table(two_by_two("image_AUROC"), "Method"),
    "Mean pixel AUROC (15 MVTec AD categories)\n\n" + md_table(two_by_two("pixel_AUROC"), "Method"),
])
(FIG_DIR / "summary_tables.md").write_text(readme_tables + "\n", encoding="utf-8")
print(readme_tables)

## 7. Findings (written for the 17 Oct 2026 run)

1. **The method matters far more than the backbone.** Replacing PaDiM with PatchCore adds about 0.07 to 0.08 image AUROC at either backbone. Replacing ResNet18 with Wide-ResNet50 adds only about 0.01 (PatchCore) to 0.02 (PaDiM).
2. **PatchCore wins almost everywhere.** At Wide-ResNet50 it is ahead on 14 of 15 categories for image AUROC (one tie, `bottle`) and on all 15 for pixel AUROC. At ResNet18 the lead is slightly smaller: 13 of 15 for image AUROC and 12 of 15 for pixel AUROC.
3. **`screw` is a robust PaDiM weakness.** PaDiM stays at about 0.69 image AUROC even with the bigger backbone, 0.27 below PatchCore.
4. **`hazelnut` is mostly a backbone effect for PaDiM.** The bigger backbone lifts it from about 0.72 to 0.89, so most of that gap closes. An earlier guess that random object rotation alone explains the `hazelnut` and `screw` gaps is not supported by the `hazelnut` result.
5. **The backbone effect for PaDiM is mixed.** It helps on 7 of 15 categories and hurts on 8. A possible explanation (not tested) is that keeping 100 of the 1,792 available features discards information that the wider backbone provides.
6. **Textures are harder to localize than objects in all four variants** (lower pixel AUROC), even though textures are as easy or easier to detect at the image level.

## 8. Caveats

- PaDiM with Wide-ResNet50 used 100 features instead of the paper's 550 (GPU memory limit). Results at 550 may differ.
- One seed per run. Differences below about 0.01 should not be interpreted, and several categories have small test sets, so a single image can move AUROC noticeably.
- All runs used anomalib's default image size and preprocessing for the installed version.
- F1 scores depend on an automatically chosen threshold and are not used for the conclusions above.